# D1.5 · Agent telemetry as a data source

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *Security of AI*

Builds on **[D1.4 · Detection engineering *for* agents](https://spbreed.github.io/cyber-commons/lessons/D1.4.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry, OpenSearch |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

You cannot detect on telemetry that was never emitted. Prompts, tool calls, decisions and identities are the four things an agent has to emit to be observable at all — and none of them appear in a standard application log.

> **At CyberTravels.** You cannot detect on what CyberTravels never emitted. Prompts, tool calls, decisions and identities are the four things missing from every application log CyberTravels has. R10, R11.

## 2 · The framework

```
   what an agent must emit to be observable at all

   +------------+  +-------------+  +-----------+  +------------+
   |  prompts   |  | tool calls  |  | decisions |  | identities |
   +------------+  +-------------+  +-----------+  +------------+
        |               |                |              |
        +---------------+----------------+--------------+
                                v
                   none of this is in an application log
                   retention is expensive and the cost is real
```

Agent telemetry has a property no other log source has: it contains the
**reasoning**, not just the action. The trace records what the agent was trying
to do, what it considered, and what the verifier said.

That is enormously useful for investigation and it is a retention and privacy
problem, because reasoning traces contain whatever was in the context window —
which routinely includes customer data, source code and secrets that were read
legitimately.

So retention has to be decided **per field**, not per record:

| Field | Forensic value | Sensitivity |
|---|---|---|
| timestamps, tool, target | high | low |
| verifier detail | high | low |
| acting identity + chain | high | low |
| model prompts | medium | **high** |
| tool results | high | **high** |

The first three are cheap and should be kept long. The last two are where the
retention conversation actually is.

## 3 · Demo — one agent run, as a telemetry record

In [ ]:
import time, hashlib
from dataclasses import dataclass, field

@dataclass
class Step:
    n: int; tool: str; target: str; verifier: str; ok: bool
    prompt: str = ""; result: str = ""

RUN = [
 Step(1, "read_file", "/work/repo/billing.py", "n/a", True,
      prompt="Investigate finding SEC-4471 in billing.py",
      result="def charge(card_number, amount):  # card_number = 4111111111111111"),
 Step(2, "search_code", "charge(", "n/a", True,
      prompt="find callers of charge()",
      result="api/checkout.py:88 charge(user.card, total)"),
 Step(3, "write_file", "/work/repo/billing.py", "tests pass", True,
      prompt="apply the fix", result="patch applied"),
]
def render(steps, fields):
    out = []
    for s in steps:
        row = {k: getattr(s, k) for k in fields}
        out.append(row)
    return out

print("full record (everything the harness saw):")
for r in render(RUN, ["n", "tool", "target", "verifier", "ok", "prompt", "result"]):
    print("   ", r)

## 4 · Where it breaks — what is actually in there

In [ ]:
import re
SENSITIVE = {
 "payment card": re.compile(r"\b4[0-9]{12}(?:[0-9]{3})?\b"),
 "email":        re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
 "aws key":      re.compile(r"\bAKIA[0-9A-Z]{16}\b"),
}
def scan_record(steps):
    hits = []
    for s in steps:
        for field in ("prompt", "result"):
            text = getattr(s, field)
            for name, pat in SENSITIVE.items():
                if pat.search(text):
                    hits.append((s.n, field, name))
    return hits

hits = scan_record(RUN)
print("sensitive content found in the trace:")
for n, field, kind in hits:
    print(f"   step {n}  {field:8s} {kind}")
print("\nNobody put a card number in the trace deliberately. The agent read a")
print("source file, and the file contained a test fixture with a real-shaped PAN.")
print("The trace is now in scope for PCI, and it is in your SIEM for 400 days.")

## 5 · The control — retention per field, and a hash for the rest

In [ ]:
RETENTION = {
 "n":        (400, "low",  "cheap, high forensic value"),
 "tool":     (400, "low",  "cheap, high forensic value"),
 "target":   (400, "low",  "path only, no contents"),
 "verifier": (400, "low",  "what the harness believed — the key forensic field"),
 "ok":       (400, "low",  ""),
 "prompt":   (30,  "high", "may contain anything the task included"),
 "result":   (7,   "high", "tool output — the highest-risk field"),
}
print(f"{'field':10s}{'days':>6}{'sensitivity':>13}  rationale")
print("-" * 74)
for f, (days, sens, why) in RETENTION.items():
    print(f"{f:10s}{days:>6}{sens:>13}  {why}")

def age_record(steps, age_days):
    """What survives after N days."""
    keep = [f for f, (d, _, _) in RETENTION.items() if d >= age_days]
    out = []
    for s in steps:
        row = {k: getattr(s, k) for k in keep}
        for f in ("prompt", "result"):
            if f not in keep and getattr(s, f):
                row[f + "_sha256"] = hashlib.sha256(
                    getattr(s, f).encode()).hexdigest()[:16]
        out.append(row)
    return out

for age in (1, 14, 90):
    aged = age_record(RUN, age)
    print(f"\nafter {age} days — step 1 record:")
    print("   ", aged[0])

In [ ]:
# Verify: the aged record is still forensically useful and no longer sensitive.
aged = age_record(RUN, 90)
class Fake:
    def __init__(self, d): self.__dict__.update(d); self.prompt = d.get("prompt",""); self.result = d.get("result","")
remaining = scan_record([Fake(r) for r in aged])
print(f"sensitive content after 90 days: {remaining or 'none'}")
assert not remaining

can_answer = all("verifier" in r and "tool" in r and "target" in r for r in aged)
print(f"can still answer 'what did it do and what did the harness believe?': {can_answer}")
assert can_answer
print("\nThe hash is retained, so if the original is recovered from a backup you")
print("can still prove it is the same content the agent saw.")

## What you just proved

The full record contains a payment-card pattern found in a source file the agent read legitimately. Per-field retention keeps timestamps, tool, target and verifier for 400 days while dropping prompts at 30 days and tool results at 7, replacing them with hashes. After 90 days no sensitive content remains and the record can still answer what the agent did and what the harness believed.

## Your turn

Check the retention period on your agent traces. If it is the same as your firewall logs, one of those two numbers was chosen without anyone looking at what the traces contain.

---

**Next → [D1.6 · Distinguishing agent from human](https://spbreed.github.io/cyber-commons/lessons/D1.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*